In [1]:
!pip install -qq -U transformers bitsandbytes
!pip install -qq -U peft trl dataset sentencepiece
!pip install -qq -U peft dataset sentencepiece
!pip install -qq -U scipy
!pip install -qq -U psutil

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 62.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.3/596.3 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 81.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 98.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 69.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 34.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# print(trl.__version__)

In [3]:
import os

os.environ['KMP_DUPLICATE_LIB_OK']='True'
os.environ['TOKENIZERS_PARALLELISM'] = 'True'

In [4]:
import time
import json
import torch
from pathlib import Path 
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling
)
from huggingface_hub import login
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import Dataset
import re
import psutil  
from typing import Optional
import tempfile
import subprocess

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [ ]:
my_token = "YOUR_HF_TOKEN"
login(token = my_token)    

# json_file_path = '/kaggle/input/data-ft-1/finetuning_data.jsonl'
json_file_path = '/kaggle/input/datasets/ayaanrzaverigmailcom/data-ft-1/FT_data.jsonl'
output_dir = "/kaggle/working/deepseek_spo"
model_name = "deepseek-ai/deepseek-coder-6.7b-instruct"

In [6]:
device = "GPU" if torch.cuda.is_available() else "CPU RAM"
print(device)

GPU


In [7]:
def load_and_prepare_data(path : str):
    print(f'Loading data from [{path}]')
    
    formatted_data = []
    with open(path, 'r', encoding = 'utf-8') as f:
        for line in f:
            item = json.loads(line)
            solution = re.sub(r'//.*', '', item['space_optimal_solution'])
            solution = re.sub(r'/\*.*?\*/', '', solution, flags=re.DOTALL)
            solution = re.sub(r'\n\s*\n', '\n', solution).strip()

            prompt = f"""<|system|>
            You are an expert C++ programmer. Provide ONLY the space-optimized DP function. NO comments, NO explanations, NO multiple solutions, NO links, NO LeetCode references, NO examples. Just the function code.

            Problem : {item['problem_name']}

            <|user|>
            {item['prompt']}
            <|end|>
            <|assistant|>
            """

            completion = f"""```cpp
            {solution}
            ```"""

            formatted_data.append({
                "text": prompt + completion,
                "problem_name": item["problem_name"],
                "prompt_raw": item["prompt"],
                "space_optimal_solution": item["space_optimal_solution"],
                "main_function": item["main_function"]
            })
    return Dataset.from_list(formatted_data)

def formatting_func(example):
    return example["prompt"] + example["completion"]

In [8]:
# full_dataset_raw = Dataset.from_list(raw_data)
full_dataset_raw = load_and_prepare_data(json_file_path)

split_dataset = full_dataset_raw.train_test_split(
    test_size=0.2, 
    seed=42,
)           

print(f"Training samples: {len(split_dataset['train'])}")
print(f"Validation samples: {len(split_dataset['test'])}")

Loading data from [/kaggle/input/datasets/ayaanrzaverigmailcom/data-ft-1/FT_data.jsonl]
Training samples: 28
Validation samples: 7


In [9]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,  # Nested quantization for better memory
    bnb_4bit_quant_type="nf4",       # Normal float 4-bit
    bnb_4bit_compute_dtype=torch.float16,
)

In [10]:
print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
    padding_side="right",
    token=my_token 
)
special_tokens = ["<|system|>", "<|user|>", "<|assistant|>", "<|end|>"]
tokenizer.add_special_tokens({"additional_special_tokens": special_tokens})

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    device_map="auto",  
    token=my_token,
    trust_remote_code=True,
)

model.resize_token_embeddings(len(tokenizer))    # after adding specialized tokens
model.config.use_cache = False  

print("Model loaded successfully!")

Loading tokenizer...


config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

Model loaded successfully!


In [11]:
model = prepare_model_for_kbit_training(model)

In [12]:
lora_config = LoraConfig(
    r=16,                          # Rank - balance between quality and speed
    lora_alpha=32,                 # Alpha scaling factor (typically 2*r)
    target_modules=["q_proj","k_proj","v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],             
    lora_dropout=0.03,             
    bias="none",                  
    task_type="CAUSAL_LM",         
)

In [13]:
training_args = TrainingArguments(
    output_dir=output_dir,
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    # Training hyperparameters
    num_train_epochs=10, 
    per_device_train_batch_size=1, 
    gradient_accumulation_steps=4, 
    per_device_eval_batch_size=1,

    # optim settings
    learning_rate=2e-5,
    weight_decay=0.01,
    max_grad_norm=1.0,
    warmup_ratio=0.1,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    
    # Memory optimization
    fp16=False,
    # bf16=False,
    gradient_checkpointing=True,
    dataloader_pin_memory = False,

    # Logging 
    logging_steps=20,
    logging_dir=f"{output_dir}/logs",
    eval_strategy="epoch",
    
    seed=42,
    report_to="none",
    ddp_find_unused_parameters=False,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [14]:
trainer = SFTTrainer(
    model = model,
    train_dataset = split_dataset['train'],
    eval_dataset = split_dataset['test'],
    args = training_args,
    peft_config = lora_config,      
    data_collator = DataCollatorForLanguageModeling(tokenizer = tokenizer, mlm = False),
    # data_collator = DataCollatorForLanguageModeling(tokenizer = tokenizer),
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Adding EOS to train dataset:   0%|          | 0/28 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/28 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/28 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

In [15]:
torch.cuda.empty_cache()

print("STARTING MODEL TRAINING")
trainer.train()
print("TRAINING COMPLETED!")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 32014}.


STARTING MODEL TRAINING


Epoch,Training Loss,Validation Loss
1,No log,1.349105
2,No log,1.267178
3,1.296864,1.178505
4,1.296864,1.092024
5,1.296864,1.011607
6,1.048991,0.947426
7,1.048991,0.899797
8,1.048991,0.876264
9,0.887174,0.869790
10,0.887174,0.869814


/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetunin

TRAINING COMPLETED!


In [16]:
final_model_path = f"{output_dir}/final_model"
trainer.model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

print(f"Model saved to: {final_model_path}")

/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


Model saved to: /kaggle/working/deepseek_spo/final_model


In [17]:
import tempfile 

TEMP_DIR = Path(tempfile.mkdtemp(prefix="dp_eval_"))

In [18]:
def extract_function(code: str) -> str:
    code = re.sub(r"```.*?```", "", code, flags=re.DOTALL)

    func_pattern = re.compile(
        r'^[\s]*(?:int|long\s+long|long|bool|void|string|double|float|char|vector<.*?>)\s+\w+\s*\([^)]*\)\s*\{',
        re.MULTILINE
    )

    match = func_pattern.search(code)
    if not match:
        return code.strip()

    start = match.start()
    brace_count = 0
    end = None

    for i in range(start, len(code)):
        if code[i] == '{':
            brace_count += 1
        elif code[i] == '}':
            brace_count -= 1
            if brace_count == 0:
                end = i + 1
                break
    if end:
        return code[start:end].strip()
    return code[start:].strip()

In [25]:
def generate_solution(problem: str, prompt: str, model, tokenizer) -> str:
    """Generate C++ solution from problem description"""
    model.eval()
    
    # Format prompt exactly as during training
    # formatted_prompt = f"""
    # <|system|>
    
    # You are an expert C++ programmer. Provide ONLY the space-optimized DP function. NO comments, NO explanations, NO multiple solutions, NO links, NO LeetCode references, NO examples. Just the function code.
    
    # Problem: {problem}
    # <|user|>
    # {prompt}
    # <|end|>
    # <|assistant|>
    # """
    formatted_prompt = f"""
    <|system|>
    You output exactly one C++ function.
    Start directly with the return type.
    Do not include explanations.
    <|user|>
    {prompt}
    <|assistant|>
    """
    
    inputs = tokenizer(
        formatted_prompt, 
        return_tensors="pt", 
        truncation=True, 
        max_length=1024,
        add_special_tokens=True
    ).to(model.device)
    
    print(f"Input shape: {inputs['input_ids'].shape}")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=600,
            temperature=0.4,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.05,
        )
    
    # Decode only the new tokens
    input_length = inputs['input_ids'].shape[1]
    generated = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    generated = extract_function(generated)
    return generated

## Calculating space used

In [20]:
def create_program_file(solution : str, main_func : str, tag : str):
    """Create complete C++ program"""
    program = f"""
    #include <bits/stdc++.h>
    using namespace std;
    
    {solution}
    
    {main_func}
    """
    file_path = TEMP_DIR / f"{tag}.cpp"
    file_path.write_text(program)
    # return program
    return file_path

In [21]:
def compile_program(cpp_file : Path) -> Optional[Path]:
    """Compile C++ program & return executable path if successful"""
    if isinstance(cpp_file, str):
        cpp_file = Path(cpp_file)
        
    exe_path = cpp_file.with_suffix('')
    result = subprocess.run(
        ['g++', '-std=c++17', '-O2', str(cpp_file), '-o', str(exe_path)], 
        capture_output=True, 
        text=True
    )
    return exe_path if result.returncode == 0 else None 

In [22]:
def measure_memory_usage(executable : Path, timeout : int = 5) -> float:
    if not executable.exists():
        return -1.0

    process = subprocess.Popen(
        [str(executable)],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True
    )

    try:
        ps_process = psutil.Process(process.pid)
        peak_mb = 0.0
        start = time.time()
        while process.poll() is None:
            if time.time() - start > timeout:
                process.terminate()
                break 
            try:
                mem_mb = ps_process.memory_info().rss / (1024 * 1024)
                peak_mb = max(peak_mb, mem_mb)
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                break
            time.sleep(0.01)
        process.wait(timeout = 2)
        return (peak_mb if peak_mb > 0.0 else -1.0)
    except Exception:
        return -1.0

In [23]:
def test_single_case(sample : dict):
    problem = sample['problem_name']
    prompt = sample['prompt_raw']
    gt_solution = sample['space_optimal_solution']
    main_func = sample['main_function']
    
    print(f"\nTESTING: {problem}")


    gen_code = generate_solution(problem, prompt, model, tokenizer)

    print("GENERATED :\n", gen_code)
    print("ORIGINAL:\n", gt_solution)
    print('\n\n')

    gt_file = create_program_file(gt_solution, main_func, "gt")
    gt_exe = compile_program(gt_file)

    gen_file = create_program_file(gen_code, main_func, "gen")
    gen_exe = compile_program(gen_file)

    if not gt_exe or not gen_exe:
        return {'problem': sample['problem_name'], 'success': False,'error': 'Compilation failed'}

    # Measure memory
    gt_mem = measure_memory_usage(gt_exe)
    gen_mem = measure_memory_usage(gen_exe)
        
    if gt_mem <= 0 or gen_mem <= 0:
        return {'problem': sample['problem_name'], 'success': False, 'error': 'Memory measurement failed (-ve mem recorded)'}

    ratio = gen_mem / gt_mem
    print(f"  ✓ GT: {gt_mem:.2f} MB | Gen: {gen_mem:.2f} MB | Ratio: {ratio:.3f}x")

In [26]:
# print(split_dataset['test'][0])

for i, sample in enumerate(split_dataset['test']):
    test_single_case(sample)


TESTING: N Lis
Input shape: torch.Size([1, 70])
GENERATED :
 int findNumberOfLIS(vector<int>& nums) {
    int n = nums.size();
    if (n == 0) return 0;
    vector<int> dp(n, 1), cnt(n, 1);
    int max_len = 1, res = 0;
    for (int i = 1; i < n; ++i) {
        for (int j = 0; j < i; ++j) {
            if (nums[i] > nums[j]) {
                if (dp[j] + 1 > dp[i]) {
                    dp[i] = dp[j] + 1;
                    cnt[i] = cnt[j]; // Important: reset count when we found a new longer subseq
                } else if (dp[j] + 1 == dp[i]) {
                    cnt[i] += cnt[j]; // accumulate counts when we found same length subseq
               
                }
            }
        }
        max_len = max(max_len, dp[i]);
    }
    for (int i = 0; i < n; ++i) {
        if (dp[i] == max_len) res += cnt[i];
    }
    return res;
}
ORIGINAL:
 int hyper_optimized_approach(vector<int> &arr){
    int n = arr.size();
    vector<int> dp(n, 1);
    vector<int> count(n, 1);
    int 